In [1]:
%%capture
!pip install datasets transformers pandas numpy seqeval evaluate torch
!pip install --upgrade transformers

In [ ]:
%%capture
!pip install -U peft bitsandbytes accelerate

In [2]:
import transformers
import datasets
import pandas as pd
import numpy as np
import evaluate
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, \
                        Trainer, TrainingArguments, DataCollatorForTokenClassification


print("Transformers library version: ", transformers.__version__)
print("Datasets library version: ", datasets.__version__)
print("Numpy version: ", np.__version__)
print("Pandas version: ", pd.__version__)
print("Torch version: ", torch.__version__)

Transformers library version:  5.5.4
Datasets library version:  4.0.0
Numpy version:  2.0.2
Pandas version:  2.2.2
Torch version:  2.10.0+cu128


# Dataset
I will use `ibm-research/MedMentions-ZS` as Dataset-1.

The ibm-research/MedMentions-ZS dataset is a specialized version of the original MedMentions corpus, specifically adapted for Zero-Shot (ZS) Named Entity Recognition (NER). It was curated by IBM Research to evaluate a model's ability to identify medical entities it has never seen during training (Zero Shot).


In [3]:
dataset = load_dataset("ibm-research/MedMentions-ZS")

# the output should contain:
# tokens - list of tokens
# ner_tags - list of tags per token
print(dataset["train"][0])

{'tokens': ['DCTN4', 'as', 'a', 'modifier', 'of', 'chronic', 'Pseudomonas', 'aeruginosa', 'infection', 'in', 'cystic', 'fibrosis'], 'ner_tags': ['B-T103', 'O', 'O', 'O', 'O', 'B-T038', 'I-T038', 'I-T038', 'I-T038', 'O', 'B-T038', 'I-T038']}


In [4]:
# Create label mappings of Named Entities
all_unique_labels = set()
for split in dataset.keys():
    for row in dataset[split]["ner_tags"]:
        all_unique_labels.update(row)
unique_labels = sorted(list(all_unique_labels))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}

# Part I: BioClinicalBERT

In this part I will work with `emilyalsentzer/Bio_ClinicalBERT`

In [6]:
model_name="emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True,
                          is_split_into_words=True, max_length=256)
    labels = []
    for i, label_list in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        # Convert string labels to integer IDs
        label_ids = [-100 if word_idx is None else label2id[label_list[word_idx]] for word_idx in word_ids]
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

tokenized_ds = dataset.map(tokenize_and_align, batched=True)

num_labels = len(unique_labels)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
)

# Freeze the first 6 layers of the BERT encoder
for layer in model.bert.encoder.layer[:6]:
    for param in layer.parameters():
        param.requires_grad = False

print("Freezed the first 6 layers of BERT.")

# Define compute_metrics function for evaluation
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    # Unpack the tuple
    logits, labels = eval_pred

    # Get the predicted IDs
    predictions = np.argmax(logits, axis=-1)

    # Convert predictions/labels from IDs to string labels
    # We ignore indices where label is -100
    true_predictions = [
        [model.config.id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [model.config.id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    print("pred: ", true_predictions)
    print("true: ", true_labels)

    # Calculate metrics
    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir="./bio-clinicalbert-medmentions-finetuned",
    per_device_train_batch_size=8,
    num_train_epochs=5,
    eval_strategy="epoch",
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    save_total_limit=1,
    save_strategy="epoch",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model("./medmentions-clinicalbert-final")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from transformers import pipeline
import os

output_dir = "./medmentions-clinicalbert-final"

# Check if the directory and config.json exist
if not os.path.exists(output_dir) or not os.path.exists(os.path.join(output_dir, 'config.json')):
    raise FileNotFoundError(f"Model directory '{output_dir}' or its config.json not found. Please ensure training completed successfully and saved the model in cell 'wSwFWTTMD1no'.")

# Load the pipeline, using the fine-tuned model directory for both model and tokenizer
nlp = pipeline("ner", model=output_dir, tokenizer=output_dir)

# Test it!
text = "The patient was diagnosed with acute myocardial infarction."
results = nlp(text)

for entity in results:
    print(f"Entity: {entity['word']}, Label: {entity['entity']}, Score: {entity['score']:.4f}")

### Comparison with Non-Fine-Tuned Model

Let's compare the results with the original `Bio_ClinicalBERT` model (non-fine-tuned) to see the impact of fine-tuning on the MedMentions-ZS dataset.

In [ ]:
# Load the non-fine-tuned Bio_ClinicalBERT model
nlp_original = pipeline("ner", model=model_name, tokenizer=model_name)

# Test it with the same text
results_original = nlp_original(text)

print("\n--- Results from Non-Fine-Tuned Model ---")
for entity in results_original:
    print(f"Entity: {entity['word']}, Label: {entity['entity']}, Score: {entity['score']:.4f}")

print("\n--- Results from Fine-Tuned Model (for comparison) ---")
for entity in results:
    print(f"Entity: {entity['word']}, Label: {entity['entity']}, Score: {entity['score']:.4f}")